# SQLBuddy 02 - LoRA fine-tuning
Fine-tunes `Qwen/Qwen2.5-Coder-1.5B-Instruct` with LoRA (r=16, alpha=32, dropout 0.05, attention + MLP
projections) on Spider train (6,648 examples; 352 from 8 held-out databases for validation).
2 epochs, lr 2e-4, cosine, effective batch 16, max length 1536, fp16, loss on the SQL completion only.
The prompt is exactly the one used for evaluation. Code: `src/train.py` in the repo.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # use one T4 (no DataParallel)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

In [ ]:
%pip uninstall -y -q torchao
%pip install -q transformers==5.22.1 peft==0.21.2 accelerate==1.15.0 datasets==5.1.0 huggingface_hub==2.4.0 nltk==3.9.2 trl==1.17.1 matplotlib==3.11.2

In [ ]:
# Locate our shared code (Kaggle dataset "sqlbuddy-src") and the Spider databases, then import.
import glob, os, shutil, sys, zipfile
work = "/kaggle/working/sqlbuddy"
# The package folder is the one containing run_eval.py + spider_eval/ (Kaggle may unpack it at the
# dataset root or keep a src/ folder); copy it to /kaggle/working/sqlbuddy/src.
hits = [h for h in glob.glob("/kaggle/input/**/run_eval.py", recursive=True)
        if os.path.isdir(os.path.join(os.path.dirname(h), "spider_eval"))]
if hits:
    shutil.copytree(os.path.dirname(hits[0]), work + "/src", dirs_exist_ok=True)
else:  # or it may still be a zip archive
    zips = glob.glob("/kaggle/input/**/src.zip", recursive=True)
    assert zips, "sqlbuddy-src dataset not attached"
    zipfile.ZipFile(zips[0]).extractall(work + "/src")
sys.path.insert(0, work)
print("src commit:", open(work + "/src/GIT_COMMIT").read().strip() if os.path.exists(work + "/src/GIT_COMMIT") else "?")

from src.data import find_spider_root
SPIDER_ROOT = find_spider_root("/kaggle/input")
print("Spider root:", SPIDER_ROOT)

import torch, transformers, peft, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__,
      "| datasets", datasets.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# MAX_STEPS = -1 -> full 2-epoch run; 30 -> smoke test to measure time/memory.
MAX_STEPS = -1
from src.train import main
args = ["--out", "/kaggle/working/train", "--spider-root", str(SPIDER_ROOT), "--max-steps", str(MAX_STEPS)]
if MAX_STEPS > 0:
    args += ["--eval-steps", "30"]
main(args)

In [ ]:
import pandas as pd
from IPython.display import Image, display
display(Image("/kaggle/working/train/loss_curve.png"))
display(pd.read_csv("/kaggle/working/train/loss_history.csv").tail(20))